In [ ]:
import pandas as pd
import numpy as np

# Use the raw GitHub URL
csv_url = "https://raw.githubusercontent.com/EdwBryan/AI-GTZAN-based-model/main/gtzan_selected_features.csv"

df = pd.read_csv(csv_url)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns")

Loaded 999 rows and 7 columns


In [ ]:
df.isin([np.inf, -np.inf]).sum()  # Ver si hay infinitos

In [ ]:
df.describe() # Ver estadísticas básicas

In [ ]:
df.isnull().sum()  # Ver si hay vacíos

In [ ]:
df['label'].value_counts() # Ver distribución de clases

In [35]:
from sklearn.model_selection import train_test_split

# Features (X) y etiqueta (y)
X = df.drop(columns=["label"])
y = df["label"]

# Split 60% train, 40% test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y,
)

print("Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")

print("\nProporciones reales:")
print(f"train: {len(X_train)/len(X):.3f}")
print(f"test : {len(X_test)/len(X):.3f}")

Shapes:
X_train: (599, 6), y_train: (599,)
X_test : (400, 6), y_test : (400,)

Proporciones reales:
train: 0.600
test : 0.400


In [36]:
from sklearn.preprocessing import StandardScaler

# Crear el escalador
scaler = StandardScaler()

# Ajustar SOLO en train (para evitar data leakage)
scaler.fit(X_train)

# Aplicar a train y test
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir a DataFrame para mantener los nombres de columnas
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Datos normalizados:")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

print("\nMedias (train - deben ser ~0):")
print(X_train_scaled.mean().round(4))

print("\nDesviación estándar (train - deben ser ~1):")
print(X_train_scaled.std().round(4))

Datos normalizados:
X_train_scaled shape: (599, 6)
X_test_scaled shape: (400, 6)

Medias (train - deben ser ~0):
mfcc_std                  -0.0
chroma_mean               -0.0
spectral_centroid_mean    -0.0
spectral_rolloff_mean      0.0
zero_crossing_rate_mean    0.0
tempo                      0.0
dtype: float64

Desviación estándar (train - deben ser ~1):
mfcc_std                   1.0008
chroma_mean                1.0008
spectral_centroid_mean     1.0008
spectral_rolloff_mean      1.0008
zero_crossing_rate_mean    1.0008
tempo                      1.0008
dtype: float64


In [37]:
# Entrenar Random Forest SOLO con train_split
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

print("\nEntrenando Random Forest con train...")
rf_model.fit(X_train_split, y_train_split)

# Predicciones
y_train_pred = rf_model.predict(X_train_split)
y_valid_pred = rf_model.predict(X_valid_split)

# Métricas
train_acc = accuracy_score(y_train_split, y_train_pred)
train_f1 = f1_score(y_train_split, y_train_pred, average='weighted')

valid_acc = accuracy_score(y_valid_split, y_valid_pred)
valid_f1 = f1_score(y_valid_split, y_valid_pred, average='weighted')

print("\n" + "="*50)
print("RANDOM FOREST - TRAIN vs VALIDACIÓN")
print("="*50)
print(f"\nAccuracy Train: {train_acc:.4f}")
print(f"F1-Score Train: {train_f1:.4f}")
print(f"\nAccuracy Valid: {valid_acc:.4f}")
print(f"F1-Score Valid: {valid_f1:.4f}")

print("\n" + "="*50)
print("REPORTE VALIDACIÓN")
print("="*50)
print(classification_report(y_valid_split, y_valid_pred))


Entrenando Random Forest con train...

RANDOM FOREST - TRAIN vs VALIDACIÓN

Accuracy Train: 0.9982
F1-Score Train: 0.9982

Accuracy Valid: 0.5357
F1-Score Valid: 0.5253

REPORTE VALIDACIÓN
              precision    recall  f1-score   support

       blues       0.40      0.29      0.33        14
   classical       0.73      0.79      0.76        14
     country       0.46      0.43      0.44        14
       disco       0.44      0.29      0.35        14
      hiphop       0.69      0.79      0.73        14
        jazz       0.40      0.43      0.41        14
       metal       0.65      0.79      0.71        14
         pop       0.67      0.71      0.69        14
      reggae       0.39      0.50      0.44        14
        rock       0.42      0.36      0.38        14

    accuracy                           0.54       140
   macro avg       0.52      0.54      0.53       140
weighted avg       0.52      0.54      0.53       140



In [38]:
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Codificar las etiquetas
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_split)
y_valid_encoded = le.transform(y_valid_split)
n_classes = len(np.unique(y_train_encoded))

print(f"Número de clases: {n_classes}")
print(f"Clases: {le.classes_}")

# Convertir a one-hot encoding
y_train_one_hot = keras.utils.to_categorical(y_train_encoded, num_classes=n_classes)
y_valid_one_hot = keras.utils.to_categorical(y_valid_encoded, num_classes=n_classes)

# Construir el modelo
model = keras.Sequential([
    layers.Input(shape=(X_train_split.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(n_classes, activation='softmax')
])

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nArquitectura del modelo:")
model.summary()

# Entrenar
print("\nEntrenando Red Neuronal...")
history = model.fit(
    X_train_split,
    y_train_one_hot,
    epochs=100,
    batch_size=32,
    validation_data=(X_valid_split, y_valid_one_hot),
    verbose=1
)

# Predicciones
y_train_pred_nn = np.argmax(model.predict(X_train_split), axis=1)
y_valid_pred_nn = np.argmax(model.predict(X_valid_split), axis=1)

# Métricas
train_acc_nn = accuracy_score(y_train_encoded, y_train_pred_nn)
train_f1_nn = f1_score(y_train_encoded, y_train_pred_nn, average='weighted')

valid_acc_nn = accuracy_score(y_valid_encoded, y_valid_pred_nn)
valid_f1_nn = f1_score(y_valid_encoded, y_valid_pred_nn, average='weighted')

print("\n" + "="*50)
print("RED NEURONAL - TRAIN vs VALIDACIÓN")
print("="*50)
print(f"\nAccuracy Train: {train_acc_nn:.4f}")
print(f"F1-Score Train: {train_f1_nn:.4f}")
print(f"\nAccuracy Valid: {valid_acc_nn:.4f}")
print(f"F1-Score Valid: {valid_f1_nn:.4f}")

print("\n" + "="*50)
print("REPORTE VALIDACIÓN")
print("="*50)
print(classification_report(y_valid_encoded, y_valid_pred_nn, target_names=le.classes_))

Número de clases: 10
Clases: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']

Arquitectura del modelo:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,562 (45.16 KB)

 Trainable params: 11,562 (45.16 KB)

 Non-trainable params: 0 (0.00 B)


Entrenando Red Neuronal...
Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.1073 - loss: 2.2971 - val_accuracy: 0.2143 - val_loss: 2.2012
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2021 - loss: 2.1793 - val_accuracy: 0.2143 - val_loss: 2.0919
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2343 - loss: 2.0802 - val_accuracy: 0.2500 - val_loss: 2.0060
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2469 - loss: 2.0409 - val_accuracy: 0.2571 - val_loss: 1.9384
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2648 - loss: 1.9803 - val_accuracy: 0.2714 - val_loss: 1.8808
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2916 - loss: 1.9147 - val_accuracy: 0.3143 - val_loss: 1.8254
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3113 - loss: 1.8868 - val_accuracy: 0.3571 - val_loss: 1.7806
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3381 - loss: 1.83